In [17]:
a = 5 ; b = 8
l = (a+(2*b))**2

In [18]:
dl_da = 2*(a+(2*b))*1
dl_db = 2*(a+(2*b))*2
dl_da,dl_db

(42, 84)

In [19]:
import torch
t_a = torch.tensor(5.0,requires_grad = True)
t_b = torch.tensor(8.0,requires_grad = True)
t_loss = ( t_a+(2*t_b))**2
print('a = ',t_a)
print('b = ',t_b)
print('loss = ',t_loss)

a =  tensor(5., requires_grad=True)
b =  tensor(8., requires_grad=True)
loss =  tensor(441., grad_fn=<PowBackward0>)


In [20]:
t_loss.backward()

In [21]:
t_loss.item()

441.0

In [22]:
t_a.grad.item(), t_b.grad.item()

(42.0, 84.0)

In [ ]:
# using matrix

In [32]:
import torch

X = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])

W1 = torch.tensor([
    [1., 0., 2., 1.],
    [0., 1., 1., 0.],
    [1., 2., 0., 1.]
], requires_grad=True)

W2 = torch.tensor([
    [1.],
    [2.],
    [1.],
    [0.]
], requires_grad=True)

Y = torch.tensor([
    [10.],
    [20.]
])


M = X @ W1
Y_hat = M @ W2

loss = ((Y - Y_hat) ** 2).sum()

print(loss.item())
loss.backward()

1565.0


In [33]:
print(W2.grad)

tensor([[ 852.],
        [1482.],
        [1074.],
        [ 852.]])


In [34]:
print(W1.grad)

tensor([[ 324.,  648.,  324.,    0.],
        [ 426.,  852.,  426.,    0.],
        [ 528., 1056.,  528.,    0.]])


In [34]:
# on the network

In [45]:
import random
random.seed(101)

BATCH_SIZE = 2
DIM_IN = 3
HIDDEN_SIZE = 4
DIM_OUT = 1

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel,self).__init__()

        self.layer_1 = torch.nn.Linear(DIM_IN,HIDDEN_SIZE,bias = False)
        self.layer_2 = torch.nn.Linear(HIDDEN_SIZE,DIM_OUT,bias = False)
        print(self.layer_1)
        print(self.layer_2)

    def forward(self,x):
        x = self.layer_1(x)
        x = self.layer_2(x)
        return x


In [53]:
import random
random.seed(101)

some_input = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])
ideal_output = torch.tensor([
    [10.],
    [20.]
])

model = TinyModel()

Linear(in_features=3, out_features=4, bias=False)
Linear(in_features=4, out_features=1, bias=False)


In [54]:
model.layer_1.weight

Parameter containing:
tensor([[-0.2876, -0.4406, -0.2225],
        [ 0.4370,  0.0841,  0.0988],
        [ 0.0538, -0.0127,  0.5624],
        [-0.0298, -0.4170,  0.4194]], requires_grad=True)

In [55]:
model.layer_1.bias

In [56]:
model.layer_2.weight

Parameter containing:
tensor([[-0.3307,  0.0418, -0.1540,  0.1073]], requires_grad=True)

In [58]:
model.layer_1.weight.grad

In [59]:
model.layer_2.weight.grad

In [61]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.2876, -0.4406, -0.2225],
         [ 0.4370,  0.0841,  0.0988],
         [ 0.0538, -0.0127,  0.5624],
         [-0.0298, -0.4170,  0.4194]], requires_grad=True),
 Parameter containing:
 tensor([[-0.3307,  0.0418, -0.1540,  0.1073]], requires_grad=True)]

In [62]:
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)
prediction = model(some_input)
prediction

tensor([[0.4231],
        [1.1563]], grad_fn=<MmBackward0>)

In [63]:
loss = (ideal_output - prediction).pow(2).sum()
loss

tensor(446.8008, grad_fn=<SumBackward0>)

In [64]:
loss.backward()

In [67]:
# we can see that the gradients have been computed for each learning weight
# but the weights remain unchanged, because we haven't run the optimizer yet.
# the optimizer is responsible for updating model weights
model.layer_1.weight

Parameter containing:
tensor([[-0.2876, -0.4406, -0.2225],
        [ 0.4370,  0.0841,  0.0988],
        [ 0.0538, -0.0127,  0.5624],
        [-0.0298, -0.4170,  0.4194]], requires_grad=True)

In [66]:
model.layer_2.weight

Parameter containing:
tensor([[-0.3307,  0.0418, -0.1540,  0.1073]], requires_grad=True)

In [68]:
optimizer.step()

In [69]:
model.layer_1.weight

Parameter containing:
tensor([[-0.3438, -0.5156, -0.3162],
        [ 0.4441,  0.0936,  0.1107],
        [ 0.0277, -0.0477,  0.5188],
        [-0.0115, -0.3927,  0.4498]], requires_grad=True)

In [71]:
model.layer_2.weight

Parameter containing:
tensor([[-0.5425,  0.1632,  0.0118,  0.1266]], requires_grad=True)

In [ ]:
# following cell shows the gradient accumulation in pytorch , if we dont zero_grad

In [72]:
print(model.layer_2.weight.grad)

for i in range(0, 5):
    prediction = model(some_input)
    loss = (ideal_output - prediction).pow(2).sum()
    loss.backward()

print(model.layer_2.weight.grad)

optimizer.zero_grad(set_to_none=False)

print(model.layer_2.weight.grad)

tensor([[ 211.8435, -121.3370, -165.7596,  -19.3167]])
tensor([[1358.6625, -675.1606, -776.6779, -178.1298]])
tensor([[0., 0., 0., 0.]])


In [73]:
# turning off autograd

def add_tensors1(x, y):
    return x + y


@torch.no_grad()
def add_tensors2(x, y):
    return x + y


a = torch.ones(2, 3, requires_grad=True) * 2
b = torch.ones(2, 3, requires_grad=True) * 3

c1 = add_tensors1(a, b)
print(c1)

c2 = add_tensors2(a, b)
print(c2)

tensor([[5., 5., 5.],
        [5., 5., 5.]], grad_fn=<AddBackward0>)
tensor([[5., 5., 5.],
        [5., 5., 5.]])


In [74]:
# creating a copy of the tensor which doesnt need the gradient tracking
x = torch.rand(5, requires_grad=True)
y = x.detach()

print(x)
print(y)

tensor([0.5710, 0.5779, 0.8260, 0.9180, 0.4195], requires_grad=True)
tensor([0.5710, 0.5779, 0.8260, 0.9180, 0.4195])


In [75]:
# creating the network - without activation

In [2]:
import torch
import random

random.seed(101)
torch.manual_seed(101)

BATCH_SIZE = 50
DIM_IN = 3
HIDDEN_SIZE = 4
DIM_OUT = 1
EPOCHS = 10
LR = 0.01

class TinyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_1 = torch.nn.Linear(DIM_IN, HIDDEN_SIZE, bias=False)
        self.layer_2 = torch.nn.Linear(HIDDEN_SIZE, DIM_OUT, bias=False)

        # print(self.layer_1)
        # print(self.layer_2)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.layer_2(x)
        return x


model = TinyModel()
num_samples = 100
X = torch.randn(num_samples, DIM_IN)
Y = (
    2 * X[:, 0]**2
    -3 * torch.sin(X[:, 1])
    +0.5 * X[:, 0] * X[:, 2]
)

Y += 0.1 * torch.randn(num_samples)
Y = Y.unsqueeze(1)

dataset = torch.utils.data.TensorDataset(X, Y)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR
)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for X_batch, Y_batch in loader:
        print(X_batch.shape)
        predictions = model(X_batch)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)

    print(f"Epoch {epoch+1:2d} | Loss = {avg_loss:.6f}")

torch.Size([50, 3])
torch.Size([50, 3])
Epoch  1 | Loss = 14.081871
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  2 | Loss = 13.976606
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  3 | Loss = 13.891689
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  4 | Loss = 13.757174
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  5 | Loss = 13.624355
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  6 | Loss = 13.498412
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  7 | Loss = 13.387649
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  8 | Loss = 13.181936
torch.Size([50, 3])
torch.Size([50, 3])
Epoch  9 | Loss = 13.015438
torch.Size([50, 3])
torch.Size([50, 3])
Epoch 10 | Loss = 12.830295
